# 04 — Exploration : pousser les embeddings dans PostgreSQL / pgvector

**Objectif** : comprendre chaque étape de l'écriture en base avant d'écrire `rag/store.py` au propre.

1. connexion via le `.env` (variables libpq) ;
2. application du schéma, puis enregistrement du type `vector` côté Python ;
3. encodage des chunks avec l'`EncodeurBGEM3` de l'incrément 2 ;
4. passage d'un `Chunk` à une ligne SQL ;
5. **upsert** (`INSERT … ON CONFLICT … DO UPDATE`) ;
6. **suppression des orphelins** et son garde-fou ;
7. synchronisation complète dans **une seule transaction** ;
8. vérifications de la base remplie ;
9. idempotence : relancer ne change rien ;
10. démonstration d'un orphelin supprimé ;
11. `EXPLAIN` : quand Postgres utilise-t-il l'index HNSW ?

Prérequis :

```bash
uv add "psycopg[binary]" pgvector
docker compose up -d
```

> Ce notebook écrit dans la base de **dev** (`chunks_w40k`). Tout est rejouable : la table se reconstruit à partir du jsonl.

## 1. Connexion

`load_dotenv` copie le contenu du `.env` dans les variables d'environnement du processus. psycopg, comme `psql`, lit **nativement** `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD` et `PGDATABASE`. D'où `psycopg.connect()` sans argument : aucun mot de passe n'apparaît dans le code.

Dans le code propre, `load_dotenv` ne sera appelé qu'au **point d'entrée** (la CLI). Les fonctions de `store.py` recevront une connexion déjà ouverte. C'est ce qui permettra aux tests de leur passer une connexion vers une base testcontainers.

Sur les transactions psycopg 3 : par défaut, une connexion ouvre implicitement une transaction à la première requête, et `with psycopg.connect() as conn:` fait un **commit** à la sortie du bloc (ou un rollback en cas d'exception). Dans un notebook, on garde la connexion ouverte et on passe en **autocommit**, pour que chaque cellule soit validée tout de suite. Les blocs `with conn.transaction():` regroupent explicitement ce qui doit réussir ou échouer ensemble.

In [1]:
import hashlib
import time
import uuid
from importlib import resources

import numpy as np
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

from assistant_regles.ingest.chunk import lire_jsonl
from assistant_regles.ingest.config import charger_config as charger_config_ingest
from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.config import charger_config as charger_config_rag
from assistant_regles.rag.embeddings import EncodeurBGEM3

load_dotenv(trouver_racine() / ".env")

conn = psycopg.connect(autocommit=True)
print("base :", conn.info.dbname, "| utilisateur :", conn.info.user,
      "| serveur :", conn.info.server_version)

base : assistant_regles | utilisateur : warhammer_admin | serveur : 180006


## 2. Schéma, puis type `vector`

On lit `schema.sql` **depuis le package** avec `importlib.resources`, exactement comme le fera `store.py`. Sans paramètres, psycopg accepte plusieurs instructions dans un seul `execute`.

**L'ordre compte** : `register_vector(conn)` demande à Postgres l'identifiant interne du type `vector`, pour que psycopg sache convertir les tableaux numpy dans ce type (et inversement). Ce type n'existe qu'une fois l'extension créée : appeler `register_vector` avant le schéma échouerait sur une base vierge, comme celles de testcontainers.

In [2]:
schema = resources.files("assistant_regles.rag").joinpath("sql/schema.sql").read_text(encoding="utf-8")

In [3]:

conn.execute(schema)          # idempotent : IF NOT EXISTS partout
register_vector(conn)

TABLE = "chunks_w40k"
print(conn.execute(f"SELECT count(*) FROM {TABLE}").fetchone()[0], "ligne(s) actuellement")

0 ligne(s) actuellement


## 3. Chunks et embeddings

On réutilise le code propre de l'incrément 2 : la config de l'ingestion donne le chemin du jsonl, et celle de l'indexation donne les paramètres du modèle.

In [4]:
charger_config_ingest().chemin_chunks

PosixPath('/home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/interim/chunks/chunks.jsonl')

In [5]:
chunks = lire_jsonl(charger_config_ingest().chemin_chunks)

In [7]:
encodeur = EncodeurBGEM3.charger(charger_config_rag().embeddings)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
debut = time.perf_counter()
vecteurs = encodeur.encoder([c.texte for c in chunks])
print(len(chunks), "chunks encodés en", f"{time.perf_counter() - debut:.1f} s ->", vecteurs.shape, vecteurs.dtype)
print("identifiant du modèle :", encodeur.identifiant)

207 chunks encodés en 0.8 s -> (207, 1024) float32
identifiant du modèle : BAAI/bge-m3@5617a9f61b028005a4858fdac845db406aefb181


## 4. D'un `Chunk` à une ligne SQL

La liste `COLONNES` est **la** correspondance entre le modèle Pydantic et la table : même ordre pour le `INSERT` et pour le tuple de valeurs.

- `id` est converti en `uuid.UUID`. Transmis comme simple texte, il serait refusé plus loin dans `id = ANY(%s)`, car Postgres ne compare pas un `uuid` à un tableau de `text`.
- Les listes Python deviennent des tableaux Postgres (`text[]`, `integer[]`), y compris les listes vides.
- Le vecteur numpy devient un `vector(1024)` grâce à `register_vector`.
- `empreinte_texte` est le sha256 du texte embarqué.

In [9]:
COLONNES = [
    "id", "texte", "chapitre", "section_num", "section_titre", "code", "sous_section",
    "sous_parties", "page_debut", "page_fin", "type_contenu", "edition", "source",
    "codes_cites", "partie", "nb_parties", "nb_tokens", "hors_budget", "ordres",
    "embedding", "modele_embedding", "empreinte_texte",
]

In [11]:
def vers_ligne(chunk, vecteur, identifiant_modele):
    """Tuple de valeurs dans l'ordre de COLONNES."""
    return (
        uuid.UUID(str(chunk.id)), chunk.texte, chunk.chapitre, chunk.section_num,
        chunk.section_titre, chunk.code, chunk.sous_section, list(chunk.sous_parties),
        chunk.page_debut, chunk.page_fin, chunk.type_contenu, chunk.edition, chunk.source,
        list(chunk.codes_cites), chunk.partie, chunk.nb_parties, chunk.nb_tokens,
        chunk.hors_budget, list(chunk.ordres), vecteur, identifiant_modele,
        hashlib.sha256(chunk.texte.encode("utf-8")).hexdigest(),
    )

In [12]:
lignes = [vers_ligne(c, v, encodeur.identifiant) for c, v in zip(chunks, vecteurs, strict=True)]
print(len(lignes), "lignes ;", len(lignes[0]), "valeurs par ligne pour", len(COLONNES), "colonnes")

207 lignes ; 22 valeurs par ligne pour 22 colonnes


## 5. L'upsert

`INSERT … ON CONFLICT (id) DO UPDATE SET …` : si l'`id` existe déjà (conflit sur la clé primaire), Postgres **met à jour** la ligne au lieu d'échouer.

`EXCLUDED` désigne la ligne qu'on **tentait** d'insérer, celle qui a été « exclue » par le conflit. `SET texte = EXCLUDED.texte` signifie donc « remplace par la nouvelle valeur ». On remet aussi `indexe_le` à `now()` pour dater la dernière écriture.

`executemany` envoie toutes les lignes en une seule série d'échanges avec le serveur (mode pipeline de psycopg 3). Pour 200 lignes, c'est largement suffisant ; `COPY` ne deviendrait intéressant qu'à partir de dizaines de milliers de lignes.

In [13]:
_liste = ", ".join(COLONNES)
_places = ", ".join(["%s"] * len(COLONNES))
_maj = ",\n    ".join(f"{c} = EXCLUDED.{c}" for c in COLONNES if c != "id")

SQL_UPSERT = f"""INSERT INTO {TABLE} ({_liste})
VALUES ({_places})
ON CONFLICT (id) DO UPDATE SET
    {_maj},
    indexe_le = now()"""

print(SQL_UPSERT)

INSERT INTO chunks_w40k (id, texte, chapitre, section_num, section_titre, code, sous_section, sous_parties, page_debut, page_fin, type_contenu, edition, source, codes_cites, partie, nb_parties, nb_tokens, hors_budget, ordres, embedding, modele_embedding, empreinte_texte)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (id) DO UPDATE SET
    texte = EXCLUDED.texte,
    chapitre = EXCLUDED.chapitre,
    section_num = EXCLUDED.section_num,
    section_titre = EXCLUDED.section_titre,
    code = EXCLUDED.code,
    sous_section = EXCLUDED.sous_section,
    sous_parties = EXCLUDED.sous_parties,
    page_debut = EXCLUDED.page_debut,
    page_fin = EXCLUDED.page_fin,
    type_contenu = EXCLUDED.type_contenu,
    edition = EXCLUDED.edition,
    source = EXCLUDED.source,
    codes_cites = EXCLUDED.codes_cites,
    partie = EXCLUDED.partie,
    nb_parties = EXCLUDED.nb_parties,
    nb_tokens = EXCLUDED.nb_tokens,
    hors_budget = EXCLUDE

## 6. Les orphelins et leur garde-fou

Un orphelin est une ligne de même `source` et même `edition` dont l'`id` n'est plus dans le jsonl (par exemple après un changement de découpage).

**Le garde-fou** : `NOT (id = ANY('{}'))` est vrai pour **toutes** les lignes. Avec une liste d'ids vide (jsonl vide, bug en amont), la requête viderait tout le périmètre. On refuse donc explicitement une liste vide.

In [14]:
SQL_ORPHELINS = f"""DELETE FROM {TABLE}
WHERE source = %s AND edition = %s AND NOT (id = ANY(%s))"""


def supprimer_orphelins(conn, source, edition, ids):
    """Supprime les lignes du périmètre absentes de `ids` ; renvoie leur nombre."""
    if not ids:
        raise ValueError("Liste d'ids vide : refus de vider tout le périmètre")
    return conn.execute(SQL_ORPHELINS, (source, edition, list(ids))).rowcount

## 7. Synchronisation complète, dans une transaction

Upsert puis suppression des orphelins, **par périmètre** (source, édition), dans `with conn.transaction():`. Si quoi que ce soit échoue, rien n'est écrit. Un lecteur ne voit jamais un état intermédiaire, par exemple les nouvelles lignes présentes mais pas encore les anciennes supprimées.

In [15]:
def synchroniser(conn, lignes):
    """Upsert de toutes les lignes puis suppression des orphelins de chaque périmètre."""
    i_id, i_source, i_edition = (COLONNES.index(c) for c in ("id", "source", "edition"))
    perimetres = {}
    for ligne in lignes:
        perimetres.setdefault((ligne[i_source], ligne[i_edition]), []).append(ligne[i_id])

    with conn.transaction():
        with conn.cursor() as cur:
            cur.executemany(SQL_UPSERT, lignes)
        supprimes = {p: supprimer_orphelins(conn, *p, ids) for p, ids in perimetres.items()}
    return supprimes


debut = time.perf_counter()
print("orphelins supprimés :", synchroniser(conn, lignes), f"({time.perf_counter() - debut:.2f} s)")

orphelins supprimés : {('livre_regles_principal', '11e'): 0} (0.31 s)


## 8. Vérifications de la base remplie

- Nombre de lignes égal au nombre de chunks.
- Dimension et norme des vecteurs, calculées **par Postgres** (`vector_dims`, `vector_norm`).
- **Aller-retour** : le vecteur relu est identique à celui envoyé. pgvector renvoie un objet `Vector`, qu'on convertit avec `.to_numpy()`.
- **Auto-récupération** : pour chaque chunk, son plus proche voisin dans la base doit être lui-même (`CROSS JOIN LATERAL` exécute la sous-requête « plus proche voisin » pour chaque ligne).

In [16]:
nb = conn.execute(f"SELECT count(*) FROM {TABLE}").fetchone()[0]
print("lignes :", nb, "| chunks :", len(chunks), "->", "OK" if nb == len(chunks) else "ÉCART")

dims, nmin, nmax = conn.execute(
    f"SELECT array_agg(DISTINCT vector_dims(embedding)), min(vector_norm(embedding)), max(vector_norm(embedding)) FROM {TABLE}"
).fetchone()
print("dimensions :", dims, f"| normes : {nmin:.6f} à {nmax:.6f}")

relu = conn.execute(f"SELECT embedding FROM {TABLE} WHERE id = %s", (lignes[0][0],)).fetchone()[0]
print("aller-retour identique :", np.array_equal(relu.to_numpy(), vecteurs[0]))

bons, total = conn.execute(f"""
    SELECT count(*) FILTER (WHERE voisin.id = c.id), count(*)
    FROM {TABLE} c
    CROSS JOIN LATERAL (
        SELECT o.id FROM {TABLE} o ORDER BY o.embedding <=> c.embedding LIMIT 1
    ) AS voisin""").fetchone()
print(f"auto-récupération : {bons}/{total}")

lignes : 207 | chunks : 207 -> OK
dimensions : [1024] | normes : 1.000000 à 1.000000
aller-retour identique : True
auto-récupération : 207/207


## 9. Idempotence

On relance exactement la même synchronisation. Attendu : même nombre de lignes, aucun orphelin, mais `indexe_le` mis à jour (l'upsert a réécrit chaque ligne).

In [17]:
avant = conn.execute(f"SELECT max(indexe_le) FROM {TABLE}").fetchone()[0]
print("orphelins supprimés :", synchroniser(conn, lignes))
apres_nb, apres = conn.execute(f"SELECT count(*), max(indexe_le) FROM {TABLE}").fetchone()
print("lignes :", apres_nb, "| indexe_le avancé :", apres > avant)

orphelins supprimés : {('livre_regles_principal', '11e'): 0}
lignes : 207 | indexe_le avancé : True


## 10. Démonstration d'un orphelin

On simule un chunk périmé : une copie d'une vraie ligne avec un **nouvel id**, dans le même périmètre. C'est ce qui arriverait après un changement de découpage. La synchronisation doit la supprimer.

In [18]:
faux_id = uuid.uuid4()
orphelin = (faux_id, *lignes[0][1:])
with conn.cursor() as cur:
    cur.execute(SQL_UPSERT, orphelin)
print("après insertion de l'orphelin :", conn.execute(f"SELECT count(*) FROM {TABLE}").fetchone()[0])

print("orphelins supprimés :", synchroniser(conn, lignes))
print("orphelin encore présent :",
      conn.execute(f"SELECT count(*) FROM {TABLE} WHERE id = %s", (faux_id,)).fetchone()[0] == 1)

try:
    supprimer_orphelins(conn, "livre_regles_principal", "11e", [])
except ValueError as e:
    print("garde-fou :", e)

après insertion de l'orphelin : 208
orphelins supprimés : {('livre_regles_principal', '11e'): 1}
orphelin encore présent : False
garde-fou : Liste d'ids vide : refus de vider tout le périmètre


## 11. `EXPLAIN` : l'index HNSW est-il utilisé ?

Pour que Postgres puisse utiliser l'index, la requête doit avoir la forme `ORDER BY embedding <=> <vecteur constant> LIMIT k`. Ici, le vecteur constant vient d'une sous-requête, évaluée une seule fois (`InitPlan`).

Sur environ 200 lignes, le planificateur peut préférer un **scan séquentiel** : lire 200 vecteurs coûte moins cher que parcourir un graphe HNSW. Ce n'est pas un bug. Pour **voir** l'index en action, on désactive temporairement les scans séquentiels.

Rappel : HNSW est une recherche **approximative**. À grande échelle, elle peut rater un vrai plus proche voisin ; `hnsw.ef_search` (40 par défaut) règle ce compromis entre rappel et vitesse.

In [19]:
REQUETE = f"""EXPLAIN (COSTS OFF)
SELECT id FROM {TABLE}
ORDER BY embedding <=> (SELECT embedding FROM {TABLE} LIMIT 1)
LIMIT 5"""

print("--- plan par défaut ---")
print("\n".join(r[0] for r in conn.execute(REQUETE)))

conn.execute("SET enable_seqscan = off")
print("\n--- scans séquentiels désactivés ---")
print("\n".join(r[0] for r in conn.execute(REQUETE)))
conn.execute("RESET enable_seqscan")

--- plan par défaut ---
Limit
  InitPlan 1
    ->  Limit
          ->  Seq Scan on chunks_w40k chunks_w40k_1
  ->  Sort
        Sort Key: ((chunks_w40k.embedding <=> (InitPlan 1).col1))
        ->  Seq Scan on chunks_w40k

--- scans séquentiels désactivés ---
Limit
  InitPlan 1
    ->  Limit
          ->  Seq Scan on chunks_w40k chunks_w40k_1
                Disabled: true
  ->  Index Scan using chunks_w40k_embedding_hnsw on chunks_w40k
        Order By: (embedding <=> (InitPlan 1).col1)


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost user=warhammer_admin database=assistant_regles) at 0x7acb70468a10>

## Bilan : ce qu'on reporte dans `rag/store.py`

- `appliquer_schema(conn)` : lecture via `importlib.resources`, puis `register_vector`, **dans cet ordre** ;
- `COLONNES` et la conversion `Chunk` → ligne ;
- `synchroniser(conn, lignes)` : upsert + orphelins par périmètre, dans une transaction, avec le garde-fou « liste vide » ;
- les vérifications de la section 8 (nombre de lignes, dimensions, normes, auto-récupération) ;
- la connexion **injectée** : `load_dotenv` seulement dans la CLI.

In [20]:
conn.close()

:) 